# Chapter 12: Cross-Validation and Model Evaluation Techniques

## 📋 Summary

Evaluating a machine learning model correctly is just as important as training it. Using a single train/test split can give misleading results — the model might get lucky (or unlucky) with one particular split. **Cross-validation** addresses this by evaluating the model on multiple splits and averaging the results, giving a more reliable estimate of generalization performance.

This chapter covers k-fold cross-validation, stratified k-fold, leave-one-out (LOO), time-series cross-validation, model selection techniques, learning curves, validation curves, and proper scoring strategies.

---

## 🧠 Theoretical Explanation

### Why Cross-Validation?
A single train/test split suffers from **high variance** in the performance estimate. Cross-validation averages performance over multiple folds, reducing variance and giving a more reliable estimate.

### K-Fold Cross-Validation
1. Split data into K equal folds
2. For each fold i: train on all folds except i, test on fold i
3. Average the K scores

Common choice: K=5 or K=10. Higher K = lower bias, higher variance, more computation.

### Stratified K-Fold
Ensures each fold has approximately the same class distribution as the full dataset. Essential for imbalanced classification problems.

### Leave-One-Out (LOO)
K = n (each sample is its own test set). Nearly unbiased but computationally expensive for large datasets.

### Time Series Cross-Validation
Standard k-fold would leak future data into training. `TimeSeriesSplit` ensures training always precedes test chronologically.

### Learning Curves
Plot training and validation score vs training set size. Diagnoses:
- **High bias (underfitting)**: Both curves plateau at low score
- **High variance (overfitting)**: Large gap between train and validation curves

### Validation Curves
Plot training and validation score vs a hyperparameter value. Shows the optimal parameter range and whether you're over/underfitting.

### Scoring Metrics
scikit-learn supports many scoring strings:
- Classification: `'accuracy'`, `'f1'`, `'roc_auc'`, `'precision'`, `'recall'`
- Regression: `'r2'`, `'neg_mean_squared_error'`, `'neg_mean_absolute_error'`


## 12.1 Basic Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Standard 5-fold CV
scores_5 = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
print(f'5-Fold CV: {scores_5.mean():.4f} (+/- {scores_5.std():.4f})')
print(f'Fold scores: {scores_5.round(4)}')

# 10-fold CV
scores_10 = cross_val_score(pipe, X, y, cv=10, scoring='accuracy')
print(f'\n10-Fold CV: {scores_10.mean():.4f} (+/- {scores_10.std():.4f})')

## 12.2 Advanced CV Methods

In [ ]:
from sklearn.model_selection import (StratifiedKFold, LeaveOneOut,
                                      RepeatedStratifiedKFold, cross_validate)

# Stratified K-Fold (preserves class distribution)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_skf = cross_val_score(pipe, X, y, cv=skf, scoring='accuracy')
print(f'Stratified 5-Fold: {scores_skf.mean():.4f} (+/- {scores_skf.std():.4f})')

# Repeated Stratified K-Fold (more stable estimate)
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scores_rskf = cross_val_score(pipe, X, y, cv=rskf, scoring='accuracy')
print(f'Repeated Stratified 5-Fold (3x): {scores_rskf.mean():.4f} (+/- {scores_rskf.std():.4f})')

# Multiple metrics at once with cross_validate
cv_results = cross_validate(pipe, X, y, cv=5,
                            scoring=['accuracy', 'f1', 'roc_auc'],
                            return_train_score=True)
print('\nMultiple metrics:')
for metric in ['test_accuracy', 'test_f1', 'test_roc_auc']:
    print(f'  {metric}: {cv_results[metric].mean():.4f}')

## 12.3 Time Series Cross-Validation

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.pyplot as plt
import numpy as np

# Visualize TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
X_ts = np.random.randn(100, 2)
y_ts = np.random.randint(0, 2, 100)

plt.figure(figsize=(12, 5))
for i, (train_idx, test_idx) in enumerate(tscv.split(X_ts)):
    plt.scatter(train_idx, [i]*len(train_idx), c='blue', s=10, label='Train' if i==0 else '')
    plt.scatter(test_idx, [i]*len(test_idx), c='red', s=10, label='Test' if i==0 else '')

plt.xlabel('Sample Index'); plt.ylabel('Fold')
plt.title('Time Series Cross-Validation Splits')
plt.legend(); plt.tight_layout(); plt.show()

# Use with a model
from sklearn.linear_model import LogisticRegression
ts_scores = cross_val_score(LogisticRegression(max_iter=200), X_ts, y_ts, cv=tscv)
print(f'Time Series CV scores: {ts_scores.round(3)}')

## 12.4 Learning Curves

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    pipe, X, y,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'b-o', label='Training score')
plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.2, color='blue')
plt.plot(train_sizes, val_mean, 'r-o', label='Validation score')
plt.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.2, color='red')
plt.xlabel('Training Set Size'); plt.ylabel('Accuracy')
plt.title('Learning Curve - Random Forest')
plt.legend(); plt.tight_layout(); plt.show()

## 12.5 Validation Curves

In [ ]:
from sklearn.model_selection import validation_curve
from sklearn.tree import DecisionTreeClassifier

param_range = range(1, 20)
train_scores_vc, val_scores_vc = validation_curve(
    DecisionTreeClassifier(random_state=42),
    X, y,
    param_name='max_depth',
    param_range=param_range,
    cv=5,
    scoring='accuracy'
)

plt.figure(figsize=(10, 6))
plt.plot(param_range, train_scores_vc.mean(axis=1), 'b-o', label='Training score')
plt.plot(param_range, val_scores_vc.mean(axis=1), 'r-o', label='Validation score')
plt.xlabel('max_depth'); plt.ylabel('Accuracy')
plt.title('Validation Curve - Decision Tree max_depth')
plt.legend(); plt.tight_layout(); plt.show()

best_depth = param_range[val_scores_vc.mean(axis=1).argmax()]
print(f'Optimal max_depth: {best_depth}')

## 12.6 Model Selection with Nested Cross-Validation

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.svm import SVC

# Nested CV: inner loop for hyperparameter tuning, outer loop for evaluation
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto']}
inner_clf = GridSearchCV(Pipeline([('s', StandardScaler()), ('svm', SVC())]),
                         {'svm__C': [0.1, 1, 10], 'svm__gamma': ['scale', 'auto']},
                         cv=inner_cv)

nested_scores = cross_val_score(inner_clf, X, y, cv=outer_cv, scoring='accuracy')
print(f'Nested CV score: {nested_scores.mean():.4f} (+/- {nested_scores.std():.4f})')

## 🔑 Key Takeaways

- **Cross-validation** gives more reliable performance estimates than a single train/test split.
- Use **StratifiedKFold** for classification to preserve class distributions in each fold.
- **Learning curves** diagnose bias/variance tradeoffs and tell you if more data would help.
- **Validation curves** show the impact of a hyperparameter and help find the sweet spot.
- **Nested cross-validation** gives an unbiased estimate when you also tune hyperparameters.
- Use `cross_validate` with multiple scoring metrics to get a comprehensive evaluation in one call.
